In [7]:
from pathlib import Path
import sys

import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from tqdm.auto import tqdm


def find_project_root(start_path: Path, required_dir: str = "src") -> Path:
    current = start_path.resolve()
    while True:
        if (current / required_dir).exists():
            return current
        if current.parent == current:
            raise FileNotFoundError(f"Cannot find project root containing '{required_dir}'")
        current = current.parent


PROJECT_ROOT = find_project_root(Path.cwd(), required_dir="src")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.datasets.multimodal_raw_dataset import MultiModalRawDataset
from src.models.multimodal.pipeline.pipeline_convnextcbam_clip_pvd import (
    MultiModalPipelineConvNeXtCBAMCLIPPVD,
)

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /media/data3/users/luongdth/MulCo-PlantNet


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


Config

In [9]:
# =====================================================
# VALIDATION SPLIT: processed validation (27 classes)
# =====================================================
VAL_ROOT = PROJECT_ROOT / "data" / "AIDG" / "dataset_PlantDoc" / "images" / "test"

# =====================================================
# TRAIN RAW SPLIT: dùng để lấy mapping class chuẩn 28 lớp
# =====================================================
TRAIN_IMAGE_ROOT = PROJECT_ROOT / "data" / "AIDG" / "dataset_PlantDoc" / "images" / "train"
TRAIN_CAPTION_ROOT = PROJECT_ROOT / "data" / "AIDG" / "captions_LLaVA" / "train"

# =====================================================
# VER1 CHECKPOINTS
# Cần sửa đúng theo file ver1 thật của bạn
# =====================================================
IMAGE_CKPT_PATH = PROJECT_ROOT / "archive" / "convnext_cbam" / "best_model.pth"
FUSION_CKPT_PATH = PROJECT_ROOT / "archive" / "multimodal_pvd" / "best_model.pth"

NUM_CLASSES = 28
BATCH_SIZE = 8
IMG_SIZE = 224
NUM_WORKERS = 0

SAVE_DIR = PROJECT_ROOT / "archive" / "ver1_validation_results_27class_safe"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = SAVE_DIR / "classification_report.csv"
CM_PATH = SAVE_DIR / "confusion_matrix.csv"
PRED_PATH = SAVE_DIR / "predictions.csv"

print("VAL_ROOT:", VAL_ROOT)
print("TRAIN_IMAGE_ROOT:", TRAIN_IMAGE_ROOT)
print("TRAIN_CAPTION_ROOT:", TRAIN_CAPTION_ROOT)
print("IMAGE_CKPT_PATH:", IMAGE_CKPT_PATH)
print("FUSION_CKPT_PATH:", FUSION_CKPT_PATH)
print("SAVE_DIR:", SAVE_DIR)

VAL_ROOT: /media/data3/users/luongdth/MulCo-PlantNet/data/AIDG/dataset_PlantDoc/images/test
TRAIN_IMAGE_ROOT: /media/data3/users/luongdth/MulCo-PlantNet/data/AIDG/dataset_PlantDoc/images/train
TRAIN_CAPTION_ROOT: /media/data3/users/luongdth/MulCo-PlantNet/data/AIDG/captions_LLaVA/train
IMAGE_CKPT_PATH: /media/data3/users/luongdth/MulCo-PlantNet/archive/convnext_cbam/best_model.pth
FUSION_CKPT_PATH: /media/data3/users/luongdth/MulCo-PlantNet/archive/multimodal_pvd/best_model.pth
SAVE_DIR: /media/data3/users/luongdth/MulCo-PlantNet/archive/ver1_validation_results_27class_safe


Helper

In [10]:
print("VAL_ROOT exists:", VAL_ROOT.exists())
print("TRAIN_IMAGE_ROOT exists:", TRAIN_IMAGE_ROOT.exists())
print("TRAIN_CAPTION_ROOT exists:", TRAIN_CAPTION_ROOT.exists())
print("IMAGE_CKPT_PATH exists:", IMAGE_CKPT_PATH.exists())
print("FUSION_CKPT_PATH exists:", FUSION_CKPT_PATH.exists())

if not VAL_ROOT.exists():
    raise FileNotFoundError(f"Validation root not found: {VAL_ROOT}")
if not TRAIN_IMAGE_ROOT.exists():
    raise FileNotFoundError(f"Train image root not found: {TRAIN_IMAGE_ROOT}")
if not TRAIN_CAPTION_ROOT.exists():
    raise FileNotFoundError(f"Train caption root not found: {TRAIN_CAPTION_ROOT}")
if not IMAGE_CKPT_PATH.exists():
    raise FileNotFoundError(f"Image checkpoint not found: {IMAGE_CKPT_PATH}")
if not FUSION_CKPT_PATH.exists():
    raise FileNotFoundError(f"Fusion checkpoint not found: {FUSION_CKPT_PATH}")

VAL_ROOT exists: True
TRAIN_IMAGE_ROOT exists: True
TRAIN_CAPTION_ROOT exists: True
IMAGE_CKPT_PATH exists: False
FUSION_CKPT_PATH exists: False


FileNotFoundError: Image checkpoint not found: /media/data3/users/luongdth/MulCo-PlantNet/archive/convnext_cbam/best_model.pth

Dataset

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

Main

In [ ]:
# =====================================================
# Mapping chuẩn từ train split 28 classes
# =====================================================
train_reference_dataset = MultiModalRawDataset(
    image_root=TRAIN_IMAGE_ROOT,
    caption_root=TRAIN_CAPTION_ROOT,
    transform=transform,
    use_depth_suppressed=False,
    strict_caption_match=True,
)

idx_to_class = train_reference_dataset.idx_to_class
class_to_idx = train_reference_dataset.class_to_idx

print("Train reference num classes:", len(class_to_idx))
print("Train reference class_to_idx:", class_to_idx)

In [ ]:
def class_name_to_text(class_name: str) -> str:
    clean_name = class_name.replace("_", " ").strip()
    return f"a photo of {clean_name}"

In [ ]:
class ValidationMultimodalDataset(Dataset):
    def __init__(self, root: Path, class_to_idx_ref: dict, transform=None):
        self.base_dataset = datasets.ImageFolder(root=str(root), transform=transform)
        self.transform = transform

        self.classes = self.base_dataset.classes
        self.class_to_idx_local = self.base_dataset.class_to_idx
        self.idx_to_class_local = {v: k for k, v in self.class_to_idx_local.items()}

        self.class_to_idx_ref = class_to_idx_ref
        self.idx_to_class_ref = {v: k for k, v in class_to_idx_ref.items()}

        # Chuyển label local của validation sang label chuẩn theo train mapping
        self.samples = []
        skipped_unknown_class = []

        for image_path, local_label in self.base_dataset.samples:
            class_name = self.idx_to_class_local[local_label]

            if class_name not in self.class_to_idx_ref:
                skipped_unknown_class.append((image_path, class_name))
                continue

            global_label = self.class_to_idx_ref[class_name]
            self.samples.append((image_path, global_label, class_name))

        print("Validation original samples:", len(self.base_dataset.samples))
        print("Validation usable samples:", len(self.samples))
        print("Validation skipped unknown classes:", len(skipped_unknown_class))
        if skipped_unknown_class:
            print("Unknown classes not in train reference:")
            for row in skipped_unknown_class:
                print(row)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, global_label, class_name = self.samples[idx]

        try:
            image = self.base_dataset.loader(image_path)
        except Exception as e:
            print(f"[ERROR] Failed to open validation image: {image_path}")
            raise e

        if self.transform is not None:
            image = self.transform(image)

        text = class_name_to_text(class_name)

        return {
            "image": image,
            "text": text,
            "label": global_label,           # label theo mapping train 28 classes
            "image_path": image_path,
            "class_name": class_name,
        }


def collate_fn(batch):
    images = torch.stack([item["image"] for item in batch], dim=0)
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    image_paths = [item["image_path"] for item in batch]
    class_names = [item["class_name"] for item in batch]

    return {
        "image": images,
        "text": texts,
        "label": labels,
        "image_path": image_paths,
        "class_name": class_names,
    }

In [ ]:
dataset = ValidationMultimodalDataset(
    root=VAL_ROOT,
    class_to_idx_ref=class_to_idx,
    transform=transform,
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=(device == "cuda"),
)

print("Validation dataset size:", len(dataset))
print("Validation local classes:", dataset.classes)

In [ ]:
def main():
    run_device = device

    print("Using device:", run_device)
    print("VAL_ROOT:", VAL_ROOT)
    print("IMAGE_CKPT_PATH:", IMAGE_CKPT_PATH)
    print("FUSION_CKPT_PATH:", FUSION_CKPT_PATH)

    try:
        model = MultiModalPipelineConvNeXtCBAMCLIPPVD(
            image_ckpt_path=IMAGE_CKPT_PATH,
            fusion_ckpt_path=FUSION_CKPT_PATH,
            num_classes=NUM_CLASSES,
            clip_model_name="ViT-L-14",
            clip_pretrained="openai",
            image_input_dim=1024,
            text_input_dim=768,
            proj_dim=512,
            proj_hidden_dim=768,
            pvd_hidden_dim=768,
            cls_hidden_dim=1024,
            dropout=0.3,
            normalize_projection=False,
            device=run_device,
        ).to(run_device)
    except torch.OutOfMemoryError:
        print("[WARN] CUDA OOM while loading model. Falling back to CPU...")
        run_device = "cpu"
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        model = MultiModalPipelineConvNeXtCBAMCLIPPVD(
            image_ckpt_path=IMAGE_CKPT_PATH,
            fusion_ckpt_path=FUSION_CKPT_PATH,
            num_classes=NUM_CLASSES,
            clip_model_name="ViT-L-14",
            clip_pretrained="openai",
            image_input_dim=1024,
            text_input_dim=768,
            proj_dim=512,
            proj_hidden_dim=768,
            pvd_hidden_dim=768,
            cls_hidden_dim=1024,
            dropout=0.3,
            normalize_projection=False,
            device=run_device,
        ).to(run_device)

    model.eval()

    y_true = []
    y_pred = []
    prediction_rows = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating ver1 on 27-class validation"):
            images = batch["image"].to(run_device, non_blocking=True)
            texts = batch["text"]
            labels = batch["label"].to(run_device, non_blocking=True)

            logits = model(images, texts)
            preds = torch.argmax(logits, dim=1)

            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

            for i in range(len(texts)):
                true_idx = labels[i].item()
                pred_idx = preds[i].item()

                prediction_rows.append({
                    "image_path": batch["image_path"][i],
                    "text": texts[i],
                    "true_label_idx": true_idx,
                    "pred_label_idx": pred_idx,
                    "true_class_name": idx_to_class[true_idx],
                    "pred_class_name": idx_to_class[pred_idx],
                    "correct": int(true_idx == pred_idx),
                })

    acc = accuracy_score(y_true, y_pred)
    print(f"Validation Accuracy: {acc:.4f}")

    labels_all = list(range(NUM_CLASSES))
    class_names_all = [idx_to_class[i] for i in labels_all]

    report = classification_report(
        y_true,
        y_pred,
        labels=labels_all,
        target_names=class_names_all,
        digits=4,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(y_true, y_pred, labels=labels_all)

    report_df = pd.DataFrame(report).transpose()
    cm_df = pd.DataFrame(cm, index=class_names_all, columns=class_names_all)
    pred_df = pd.DataFrame(prediction_rows)

    report_df.to_csv(REPORT_PATH, encoding="utf-8-sig")
    cm_df.to_csv(CM_PATH, encoding="utf-8-sig")
    pred_df.to_csv(PRED_PATH, index=False, encoding="utf-8-sig")

    print("Saved classification report to:", REPORT_PATH)
    print("Saved confusion matrix to:", CM_PATH)
    print("Saved predictions to:", PRED_PATH)

In [ ]:
main()